# Part 1 — Repository Mining
**LOG6307E · Mining Human–AI Collaboration in Software Engineering Repositories**

**Source:** [`SAILResearch/AI_Teammates_in_SE3`](https://github.com/SAILResearch/AI_Teammates_in_SE3) (replication package of the *AIDev* study).
The GitHub repository only contains scripts, figures and PDFs. The mined data itself is published as Parquet files on HuggingFace
([`hao-li/AIDev`](https://huggingface.co/datasets/hao-li/AIDev)), which is what the repository's own `load_AIDev.ipynb` loads. So **"mining" here means**
extracting, joining and aggregating those published tables into one analysis-ready dataset. We do not re-crawl the GitHub API.

**What this notebook produces** (all written to `data/`):

| File | Granularity | Content |
|---|---|---|
| `mined_pull_requests.csv` | 1 row / PR | ID, title, body, author, agent, dates, state, labels, task type, repo metadata, comment/review/commit/file counts |
| `mined_pr_files.csv.gz` | 1 row / (PR, file) | file-level additions, deletions, status |
| `mined_pr_text.csv.gz` | 1 row / comment, review or review comment | author, timestamp, text |
| `mining_manifest.json` | – | dataset revision, row counts, config, library versions |

**Scope decision:** we use the **AIDev-pop** subset (repositories with > 100 stars), the only subset that ships with comments, reviews, commits and file-level data, plus its matching Human-PR baseline.

## 0. Setup

In [1]:
# pip install -r requirements.txt   (pandas, pyarrow, fsspec, huggingface_hub, requests)
import os, json, platform, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50, "display.width", 160)

# ---- Configuration -----------------------------------------------------------
HF_REPO     = "hao-li/AIDev"
from huggingface_hub import HfApi
HF_REVISION = next(c.commit_id for c in HfApi().list_repo_commits("hao-li/AIDev", repo_type="dataset")
                   if c.commit_id.startswith("68ed5f4b80"))
print("Using dataset revision:", HF_REVISION)       # pinned to an exact commit sha in the manifest below
SEED        = 42
SAMPLE_PER_GROUP = None       # e.g. 2000 for a fast dry-run; None = full dataset
CACHE_RAW   = True            # keep a local copy of downloaded tables in data/raw/

ROOT = Path(os.environ.get("MSR_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
RAW_DIR  = ROOT / "data" / "raw"
DATA_DIR = ROOT / "data"
for d in (RAW_DIR, DATA_DIR, ROOT / "figures"):
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

Using dataset revision: 68ed5f4b80d27a9e057fc57567f38bd322ac73ec
Project root: c:\Vakken_TUE\Release Engineering\release_engineering_reproduction_project\Reproduction Project


## 1. Data access helper
`read_table` first looks for a cached copy in `data/raw/`; otherwise it streams the Parquet file from HuggingFace, reading **only the columns we need**
(this matters for `pr_commit_details`, which contains full patches and is very large; we never load the `patch` column).

In [2]:
import hashlib

def read_table(name, columns=None, filters=None):
    tag = hashlib.md5(repr((HF_REVISION, columns, filters)).encode()).hexdigest()[:8]
    local = RAW_DIR / f"{name}__{tag}.parquet"
    from_hf = not local.exists()
    src = local if not from_hf else f"hf://datasets/{HF_REPO}@{HF_REVISION}/{name}.parquet"
    try:
        df = pd.read_parquet(src, columns=columns, filters=filters)
    except Exception as e:
        print(f"  [warn] {name}: {type(e).__name__} -> retrying without column/filter pushdown")
        df = pd.read_parquet(src)
        if columns is not None:
            df = df[[c for c in columns if c in df.columns]]
        if filters is not None:   # apply simple ("col", "in", values) filters in pandas
            for col, op, vals in filters:
                if op == "in" and col in df.columns:
                    df = df[df[col].isin(vals)]
    if from_hf and CACHE_RAW:
        df.to_parquet(local, index=False)
    print(f"  loaded {name:<24} {len(df):>10,} rows x {df.shape[1]:>2} cols  ({'cache' if not from_hf else 'HuggingFace'})")
    return df

def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True)   # 'None' / '' / None -> NaT

# Record the exact dataset commit for reproducibility
try:
    from huggingface_hub import HfApi
    HF_SHA = HfApi().dataset_info(HF_REPO, revision=HF_REVISION).sha
except Exception as e:
    HF_SHA = f"unknown ({type(e).__name__})"
print("HuggingFace dataset revision:", HF_SHA)

HuggingFace dataset revision: 68ed5f4b80d27a9e057fc57567f38bd322ac73ec


## 2. Load pull requests (AI agents + human baseline)
* `pull_request` → Agentic-PRs from five agents (`agent` ∈ OpenAI Codex, Devin, GitHub Copilot, Cursor, Claude Code).
* `human_pull_request` → matched baseline of human-authored PRs, which we label `agent = "Human"`.

In [3]:
PR_COLS = ["id","number","title","body","agent","user_id","user","state",
           "created_at","closed_at","merged_at","repo_id","repo_url","html_url"]

pr_ai = read_table("pull_request", PR_COLS)
pr_hu = read_table("human_pull_request", [c for c in PR_COLS if c != "agent"])   # baseline table has no agent column
pr_ai["source_table"] = "pull_request"
pr_hu["source_table"] = "human_pull_request"
pr_hu["agent"] = "Human"

assert {"id","agent","created_at","merged_at","repo_id"} <= set(pr_ai.columns), "unexpected pull_request schema"
print("\nAI agent values:", sorted(pr_ai["agent"].dropna().unique()))
print("PR-id overlap between AI and Human tables:", len(set(pr_ai["id"]) & set(pr_hu["id"])))

  loaded pull_request                 33,596 rows x 14 cols  (HuggingFace)
  [warn] human_pull_request: ArrowInvalid -> retrying without column/filter pushdown
  loaded human_pull_request            6,618 rows x 12 cols  (HuggingFace)

AI agent values: ['Claude_Code', 'Copilot', 'Cursor', 'Devin', 'OpenAI_Codex']
PR-id overlap between AI and Human tables: 0


In [4]:
pr = pd.concat([pr_ai, pr_hu], ignore_index=True)
n_before = len(pr)
pr = pr.drop_duplicates(subset="id", keep="first")       # PR id is the primary key
print(f"Duplicate PR ids removed: {n_before - len(pr)}")

pr = pr.rename(columns={"id": "pr_id", "number": "pr_number", "user": "author"})
for c in ["created_at", "closed_at", "merged_at"]:
    pr[c] = to_dt(pr[c])
pr["state"]  = pr["state"].astype(str).str.lower()
pr["is_ai"]  = pr["agent"].ne("Human")
pr["merged"] = pr["merged_at"].notna()

# Optional down-sampling for fast dry-runs (stratified by agent group, reproducible)
if SAMPLE_PER_GROUP:
    pr = (pr.groupby("is_ai", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_GROUP), random_state=SEED)))
    print(f"[SAMPLE MODE] kept {len(pr):,} PRs")
pr_ids = set(pr["pr_id"])
pr.groupby("agent").size().rename("n_prs").to_frame()

Duplicate PR ids removed: 0


,n_prs
agent,
Claude_Code,459
Copilot,4970
Cursor,1541
Devin,4827
Human,6618
OpenAI_Codex,21799


### 2.1 Repository metadata (language, stars)

In [5]:
pr = pr.drop(columns=[c for c in ["repo_full_name","repo_language","repo_stars","repo_forks"] if c in pr.columns])
repo = read_table("repository", ["id","full_name","language","stars","forks"])
repo = repo.rename(columns={"id": "repo_id", "full_name": "repo_full_name",
                            "language": "repo_language", "stars": "repo_stars", "forks": "repo_forks"})
repo = repo.drop_duplicates("repo_id")
repo["repo_id"] = repo["repo_id"].astype("Int64")

# Human PRs have no repo_id: recover it from the owner/repo part of the URL
name_to_id = repo.assign(k=repo["repo_full_name"].str.lower()).set_index("k")["repo_id"]
url = pr["repo_url"].fillna(pr["html_url"])
key = url.str.extract(r"github\.com/(?:repos/)?([^/]+/[^/#?]+)", expand=False).str.lower()
pr["repo_id"] = pr["repo_id"].astype("Int64").fillna(key.map(name_to_id).astype("Int64"))

pr = pr.merge(repo, on="repo_id", how="left", validate="m:1")
print("Share of PRs with repository metadata, by group:")
print(pr.groupby("is_ai")["repo_full_name"].apply(lambda s: s.notna().mean()).rename({True: "AI", False: "Human"}))

  loaded repository                    2,807 rows x  5 cols  (HuggingFace)
Share of PRs with repository metadata, by group:
is_ai
Human    0.992596
AI       1.000000
Name: repo_full_name, dtype: float64


### 2.2 Task type
Both tables hold one task-type label per PR (conventional-commit style: `feat`, `fix`, `docs`, `refactor`, `test`, `ci`, `build`, `chore`, …),
assigned by the AIDev authors (LLM-based classifier). We reuse these labels and **audit** them in Part 2.

In [6]:
pr = pr.drop(columns=["task_type"], errors="ignore")   # safe to rerun

tt = pd.concat([read_table("pr_task_type"), read_table("human_pr_task_type")], ignore_index=True)
tt = (tt.rename(columns={"id": "pr_id", "type": "task_type"})[["pr_id", "task_type"]]
        .drop_duplicates("pr_id"))
pr = pr.merge(tt, on="pr_id", how="left")

print("PRs without a task type, by group:")
print(pr.groupby("is_ai")["task_type"].apply(lambda s: s.isna().sum()).rename({True: "AI", False: "Human"}))
pr["task_type"].value_counts(dropna=False).head(15)

  loaded pr_task_type                 33,596 rows x  6 cols  (HuggingFace)
  loaded human_pr_task_type            6,618 rows x  6 cols  (HuggingFace)
PRs without a task type, by group:
is_ai
Human    0
AI       0
Name: task_type, dtype: int64


task_type
feat        16396
fix          9884
docs         4406
refactor     2657
test         2539
chore        1743
build        1241
ci            511
perf          428
style         247
other         146
revert         16
Name: count, dtype: int64

### 2.3 Labels
The `pull_request` table has no labels column. Labels live in `pr_timeline` as `labeled` / `unlabeled` events.
We replay the events in chronological order to obtain the **final** label set of each PR.

In [7]:
print("Event types present in pr_timeline (cheap single-column read):")
print(read_table("pr_timeline", ["event"])["event"].value_counts().head(25))

Event types present in pr_timeline (cheap single-column read):
  loaded pr_timeline                 325,500 rows x  1 cols  (HuggingFace)
event
committed                        79735
commented                        32957
closed                           29677
labeled                          28415
merged                           22672
head_ref_deleted                 19292
review_requested                 16796
reviewed                         15865
deployed                         15089
assigned                         11627
copilot_work_started             10373
copilot_work_finished             9171
renamed                           6459
mentioned                         6307
subscribed                        6201
ready_for_review                  3372
referenced                        2010
head_ref_force_pushed             1751
unlabeled                          870
auto_squash_enabled                684
review_request_removed             589
added_to_merge_queue               54

In [8]:
LABEL_EVENTS = {"labeled", "unlabeled"}
ev = read_table("pr_timeline", ["pr_id","event","label","created_at"],
                filters=[("event", "in", sorted(LABEL_EVENTS))])
ev = ev[ev["event"].isin(LABEL_EVENTS) & ev["pr_id"].isin(pr_ids)].copy()
ev["created_at"] = to_dt(ev["created_at"])
ev = ev.sort_values(["pr_id", "created_at"], kind="stable")

state = {}
for pid, e, lab in zip(ev["pr_id"], ev["event"], ev["label"]):
    s = state.setdefault(pid, set())
    if isinstance(lab, str):
        (s.add if e == "labeled" else s.discard)(lab)

pr["labels"]   = pr["pr_id"].map(lambda p: ";".join(sorted(state.get(p, ()))))
pr["n_labels"] = pr["labels"].map(lambda s: len(s.split(";")) if s else 0)
print(f"{(pr['n_labels'] > 0).mean():.1%} of PRs carry at least one label")

  loaded pr_timeline                  29,285 rows x  4 cols  (HuggingFace)
57.6% of PRs carry at least one label


## 3. Communication data: comments, reviews, review comments
Three kinds of discussion text exist on a PR. They are kept separate in the text table (`kind` column):

* `pr_comment`: general conversation thread
* `review`: a submitted review (state `APPROVED` / `CHANGES_REQUESTED` / `COMMENTED`, optional body)
* `review_comment`: inline comment on the diff. This table is keyed to a *review*, not a PR, so we map `pull_request_review_id → pr_id` through `pr_reviews`.

In [9]:
comments = read_table("pr_comments", ["id","pr_id","user","created_at","body"])
reviews  = read_table("pr_reviews",  ["id","pr_id","user","state","submitted_at","body"])
rcom     = read_table("pr_review_comments_v2",
                      ["id","pull_request_review_id","user","path","created_at","body","in_reply_to_id"])

comments = comments[comments["pr_id"].isin(pr_ids)].drop_duplicates("id")
reviews  = reviews[reviews["pr_id"].isin(pr_ids)].drop_duplicates("id")

rcom = rcom.merge(reviews[["id","pr_id"]].rename(columns={"id": "pull_request_review_id"}),
                  on="pull_request_review_id", how="left")
print("Review comments that could not be mapped to a PR in scope:", rcom["pr_id"].isna().sum())
rcom = rcom[rcom["pr_id"].isin(pr_ids)].drop_duplicates("id")

  loaded pr_comments                  39,122 rows x  5 cols  (HuggingFace)
  loaded pr_reviews                   28,875 rows x  6 cols  (HuggingFace)
  loaded pr_review_comments_v2        26,868 rows x  7 cols  (HuggingFace)
Review comments that could not be mapped to a PR in scope: 0


In [10]:
# One long table of all discussion text
text = pd.concat([
    comments.assign(kind="pr_comment",     created_at=to_dt(comments["created_at"]))[["pr_id","kind","id","user","created_at","body"]],
    reviews .assign(kind="review",         created_at=to_dt(reviews["submitted_at"]),
                    body=reviews["body"])[["pr_id","kind","id","user","created_at","body","state"]],
    rcom    .assign(kind="review_comment", created_at=to_dt(rcom["created_at"]))[["pr_id","kind","id","user","created_at","body","path"]],
], ignore_index=True).rename(columns={"id": "item_id", "state": "review_state", "path": "file_path"})
text["pr_id"] = text["pr_id"].astype("int64")

# Per-PR discussion counts. 0 is a legitimate value here (a PR can have no comments).
def count(df): return pr["pr_id"].map(df.groupby("pr_id").size()).fillna(0).astype(int)
pr["n_comments"]        = count(comments)
pr["n_reviews"]         = count(reviews)
pr["n_review_comments"] = count(rcom)

# Distinct reviewers (excluding the PR author)
auth = pr.set_index("pr_id")["author"].str.lower()
rv = reviews.assign(a=reviews["pr_id"].map(auth))
rv = rv[rv["user"].str.lower() != rv["a"]]
pr["n_reviewers"] = pr["pr_id"].map(rv.groupby("pr_id")["user"].nunique()).fillna(0).astype(int)

# Time of first reaction by someone other than the PR author (any comment / review / review comment)
t = text.assign(a=text["pr_id"].map(auth))
t = t[t["user"].str.lower() != t["a"]]
pr["first_response_at"] = pr["pr_id"].map(t.groupby("pr_id")["created_at"].min())
print(pr[["n_comments","n_reviews","n_review_comments","n_reviewers"]].describe().T[["mean","50%","max"]])

                       mean  50%    max
n_comments         0.972845  0.0   30.0
n_reviews          0.718034  0.0   30.0
n_review_comments  0.668126  0.0  159.0
n_reviewers        0.334908  0.0   16.0


## 4. Code-change data: commits and file-level changes
`pr_commit_details` has one row per *(commit, file)*. A file edited in several commits of the same PR appears several times,
so we aggregate to a **(PR, file)** table first. The PR-level `additions`/`deletions` are then sums of per-commit line changes, i.e. **total churn** across the PR's commits, not the net diff.
The `patch` column is deliberately not loaded.

In [11]:
commits = read_table("pr_commits", ["sha","pr_id","author","committer","message"])
commits = commits[commits["pr_id"].isin(pr_ids)].drop_duplicates(["sha","pr_id"])
pr["n_commits"] = pr["pr_id"].map(commits.groupby("pr_id")["sha"].nunique())   # NaN = no commit data

det = read_table("pr_commit_details",
                 ["sha","pr_id","filename","status","additions","deletions","changes"])   # no 'patch'
det = det[det["pr_id"].isin(pr_ids)].drop_duplicates(["sha","pr_id","filename"])

files = (det.groupby(["pr_id","filename"], as_index=False)
            .agg(additions=("additions","sum"), deletions=("deletions","sum"),
                 n_commits_touching=("sha","nunique"), last_status=("status","last")))
files["churn"] = files["additions"] + files["deletions"]
files["extension"] = files["filename"].str.extract(r"(\.[A-Za-z0-9]+)$", expand=False).str.lower()

agg = files.groupby("pr_id").agg(n_files=("filename","size"),
                                 additions=("additions","sum"),
                                 deletions=("deletions","sum"))
pr = pr.merge(agg, left_on="pr_id", right_index=True, how="left")   # NaN = no file data
pr["churn"] = pr["additions"] + pr["deletions"]
print(f"PRs with file-level data: {pr['n_files'].notna().mean():.1%}")

  loaded pr_commits                   88,576 rows x  5 cols  (HuggingFace)
  loaded pr_commit_details           711,923 rows x  7 cols  (HuggingFace)
PRs with file-level data: 82.3%


## 5. Sanity checks (read these before trusting anything downstream)
The most important check is **side-table coverage by group**. If human PRs systematically lack comments/commits/files compared with AI PRs,
a naive comparison would be biased by *missing data*, not by behaviour, and we must restrict those analyses to PRs that have data.

In [12]:
cov = pd.DataFrame({
    "n_prs":               pr.groupby("agent").size(),
    "has_commit_data":     pr.groupby("agent")["n_commits"].apply(lambda s: s.notna().mean()),
    "has_file_data":       pr.groupby("agent")["n_files"].apply(lambda s: s.notna().mean()),
    "has_any_discussion":  pr.groupby("agent").apply(lambda g: ((g.n_comments + g.n_reviews + g.n_review_comments) > 0).mean()),
    "has_task_type":       pr.groupby("agent")["task_type"].apply(lambda s: s.notna().mean()),
    "has_labels":          pr.groupby("agent")["n_labels"].apply(lambda s: (s > 0).mean()),
    "merged_rate":         pr.groupby("agent")["merged"].mean(),
})
cov.style.format({c: "{:.1%}" for c in cov.columns if c != "n_prs"})

,n_prs,has_commit_data,has_file_data,has_any_discussion,has_task_type,has_labels,merged_rate
agent,,,,,,,
Claude_Code,459,99.8%,99.6%,69.1%,100.0%,20.0%,59.0%
Copilot,4970,99.9%,90.6%,76.9%,100.0%,15.9%,43.0%
Cursor,1541,99.9%,99.9%,78.8%,100.0%,21.9%,65.2%
Devin,4827,99.9%,99.8%,99.7%,100.0%,15.7%,53.8%
Human,6618,0.0%,0.0%,0.0%,100.0%,0.0%,76.8%
OpenAI_Codex,21799,100.0%,100.0%,20.8%,100.0%,97.3%,82.6%


In [30]:
core = ["pr_id","pr_number","title","author","created_at","closed_at","merged_at","labels","n_comments","n_reviews","body","n_files"]
miss = pr[core].isna().mean().rename("missing_share").to_frame()
miss["note"] = ["", "", "", "", "", "open PRs (expected)", "unmerged PRs (expected)", "empty = no labels", "", "", "empty PR description", "no file data"][:len(miss)]
print("Date range:", pr["created_at"].min(), "->", pr["created_at"].max())
print("created_at <= closed_at violated:", (pr["closed_at"] < pr["created_at"]).sum(), "| merged_at without closed_at:", (pr["merged_at"].notna() & pr["closed_at"].isna()).sum())
miss.style.format({"missing_share": "{:.1%}"})

Date range: 2024-12-24 17:57:41+00:00 -> 2025-07-30 14:44:55+00:00
created_at <= closed_at violated: 0 | merged_at without closed_at: 0


,missing_share,note
pr_id,0.0%,
pr_number,0.0%,
title,0.0%,
author,0.0%,
created_at,0.0%,
closed_at,7.1%,open PRs (expected)
merged_at,24.4%,unmerged PRs (expected)
labels,0.0%,empty = no labels
n_comments,0.0%,
n_reviews,0.0%,


## 6. Save the mined dataset

In [13]:
pr_cols = ["pr_id","pr_number","title","body","author","agent","is_ai","state","merged",
           "created_at","closed_at","merged_at","first_response_at",
           "labels","n_labels","task_type",
           "n_comments","n_reviews","n_review_comments","n_reviewers",
           "n_commits","n_files","additions","deletions","churn",
           "repo_id","repo_full_name","repo_language","repo_stars","repo_forks","html_url"]
pr_out = pr[pr_cols].sort_values(["agent","created_at"])

text_out  = text[text["pr_id"].isin(pr_ids)].sort_values(["pr_id","created_at"])
files_out = files.sort_values(["pr_id","filename"])

pr_out.to_csv(DATA_DIR / "mined_pull_requests.csv", index=False)
files_out.to_csv(DATA_DIR / "mined_pr_files.csv.gz", index=False)
text_out.to_csv(DATA_DIR / "mined_pr_text.csv.gz", index=False)

manifest = {
    "mined_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "source": {"github": "SAILResearch/AI_Teammates_in_SE3", "dataset": HF_REPO,
               "revision_requested": HF_REVISION, "revision_sha": HF_SHA, "subset": "AIDev-pop (>100 stars)"},
    "config": {"seed": SEED, "sample_per_group": SAMPLE_PER_GROUP},
    "rows": {"pull_requests": len(pr_out), "pr_files": len(files_out), "text_items": len(text_out),
             "prs_by_agent": pr_out["agent"].value_counts().to_dict()},
    "env": {"python": platform.python_version(), "pandas": pd.__version__},
}
(DATA_DIR / "mining_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest["rows"], indent=2))

{
  "pull_requests": 40214,
  "pr_files": 575913,
  "text_items": 94865,
  "prs_by_agent": {
    "OpenAI_Codex": 21799,
    "Human": 6618,
    "Copilot": 4970,
    "Devin": 4827,
    "Cursor": 1541,
    "Claude_Code": 459
  }
}
